In [77]:
import polars as pl
import json
import pandas as pd

pd.options.display.max_columns= None
pd.options.display.max_colwidth= None
pd.options.display.max_rows = None



# # Load CSV
# df = pl.read_csv("/home/mahdi/Loan_Reommender_System/wizard_solution/MEC-LoanRecomn_Scenarios-V0.7.csv", null_values=["nan"])


# ## delete Nezam Mohandesi and Garduneh
# df = df[:-6]

df = pl.read_csv("MEC-LoanRecomn_Scenarios-V14.csv")
# Convert to list of dictionaries (row-oriented)
loans = df.to_dicts()
print(len(loans))

# Dump to JSON string
# json_str = json.dumps(loans, indent=4)

# # Save to file
# with open("output.json", "w") as list_loans:
#     list_loans.write(json_str)



182


In [78]:
df.tail()

id,nickname,package_name,contract_type,granted_method,loan_amount_limit,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,minimum_deposit_amount,maximum_deposit_amount,minimum_loan_amount,guarantee,receiving_channel
i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,str,i64,i64,str
178,"""شایان""","""شایان یک""","""مرابحه/جعاله""","""واریز به حساب""",1000000000,6,23,24,125,"""N""","""nan""","""15000000000""",100000000,1,"""غیر حضوری/ حضوری"""
179,"""شایان""","""شایان یک""","""مرابحه""","""کارت اعتباری""",1000000000,6,23,24,150,"""N""","""nan""","""15000000000""",100000000,1,"""غیر حضوری/ حضوری"""
180,"""شایان""","""شایان یک""","""مرابحه/جعاله""","""واریز به حساب""",1000000000,6,18,36,40,"""N""","""nan""","""15000000000""",100000000,1,"""غیر حضوری/ حضوری"""
181,"""شایان""","""شایان یک""","""مرابحه/جعاله""","""واریز به حساب""",1000000000,6,23,36,90,"""N""","""nan""","""15000000000""",100000000,1,"""غیر حضوری/ حضوری"""
182,"""شایان""","""شایان یک""","""مرابحه""","""کارت اعتباری""",1000000000,6,23,36,100,"""N""","""nan""","""15000000000""",100000000,1,"""غیر حضوری/ حضوری"""


In [79]:
# df.info()

In [80]:
def calculate_monthly_repayment(loan_amount, interest_rate, repayment_duration):

    # Convert annual interest rate to monthly interest rate (decimal)
    # monthly_repayment = (4 / (12*100))
    # interest_rate =23
    monthly_interest_rate = (interest_rate / (12*100))
    

    # Apply the formula from the Excel file
    numerator = loan_amount * monthly_interest_rate * ((1 + monthly_interest_rate) ** repayment_duration)
    denominator = ((1 + monthly_interest_rate) ** repayment_duration) - 1
    monthly_repayment = numerator / denominator
    # monthly_repayment = monthly_interest_rate 

    return round(monthly_repayment)

# Example usage
loan_amount = 50000  # 1 billion
interest_rate = 23  # 23%
repayment_duration = 12  # 36 months

monthly_repayment = calculate_monthly_repayment(loan_amount, interest_rate, repayment_duration)


print(f"Monthly repayment amount: {monthly_repayment}")
# print(f"Total repayment amount: {total_repayment:,.2f}")

Monthly repayment amount: 4704


In [81]:
(23 / (12*100))

0.019166666666666665

In [82]:
def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                 loan_amount=None, credit_score=None, interest_rate=None, 
                 repayment_duration=None):
    
    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }

    filtered_loans = []

    for loan in loans:
        loan_copy = loan.copy()  # Avoid mutating the original loan data
        
        # Step 1: Apply the loan_amount logic
        if loan_amount is not None:
            loan_copy["loan_amount"] = loan_amount
            
            # Check that loan_coefficient exists and is not None or zero
            coefficient = loan_copy.get('loan_coefficient')
            rd = loan_copy.get('repayment_duration')
            ir = loan_copy.get('interest_rate')
            la = loan_copy.get('loan_amount')


            if coefficient:
                loan_copy["deposit_amount"] = loan_amount / coefficient
            else:
                loan_copy["deposit_amount"] = None  # Or set to 0 or skip, depending on your logic

            loan_copy["repayment_amount"] = calculate_monthly_repayment(la, ir, rd)

        # Step 2: Check conditions
        conditions = []  
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    # Check if loan's limit is enough
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                else:
                    conditions.append(loan_copy.get(key) == value)

        if conditions and all(conditions):
            filtered_loans.append(loan_copy)
    
    return filtered_loans


In [83]:
filtered = filter_loans(loans, loan_amount= 1000000000)
print(len(filtered))
print("Filtered loans:")
print(filtered[1])
# for loan in filtered:
#     print(loan)

182
Filtered loans:
{'id': 2, 'nickname': 'بهان(بدون ضامن)', 'package_name': 'فرابانک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 1000000000, 'deposit_duration': 3, 'interest_rate': 23, 'repayment_duration': 60, 'loan_coefficient': 1000, 'credit_score': 'B', 'minimum_deposit_amount': 'nan', 'maximum_deposit_amount': 'nan', 'minimum_loan_amount': 100000000, 'guarantee': 0, 'receiving_channel': 'غیر حضوری', 'loan_amount': 1000000000, 'deposit_amount': 1000000.0, 'repayment_amount': 28190471}


## Fill features first 

In [84]:
def fill_calculated_param (loans, loan_amount = None, deposit_amount= None):
    
    updated_loans = []
    
    for loan in loans:

        coefficient = loan.get('loan_coefficient')

        if (loan_amount is not None and deposit_amount is not None) or (loan_amount is not None and deposit_amount is None):
            loan["loan_amount"] = loan_amount
            if coefficient:
                loan["deposit_amount"] = round(loan_amount / coefficient)
            else:
                loan["deposit_amount"] = None

        elif deposit_amount is not None and loan_amount is None:
            loan["deposit_amount"] = deposit_amount
            if coefficient:
                loan["loan_amount"] = round(deposit_amount * coefficient)
            else:
                loan["loan_amount"] = None
            
        else:
            loan["loan_amount"] = None
            loan["deposit_amount"] = None
        
        rd = loan.get('repayment_duration')
        ir = loan.get('interest_rate')
        la = loan.get('loan_amount')

        if la is not None and ir is not None and rd is not None:
            loan["repayment_amount"] = calculate_monthly_repayment(la, ir, rd)
        else:
            loan["repayment_amount"] = None

        

        updated_loans.append(loan)

    return updated_loans

In [85]:
filtered = fill_calculated_param(loans, loan_amount= 100_000_000, deposit_amount= 120_000)
print(len(filtered))
print("Filtered loans:")
print(filtered[1])
for loan in filtered:
    print(loan)

182
Filtered loans:
{'id': 2, 'nickname': 'بهان(بدون ضامن)', 'package_name': 'فرابانک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 1000000000, 'deposit_duration': 3, 'interest_rate': 23, 'repayment_duration': 60, 'loan_coefficient': 1000, 'credit_score': 'B', 'minimum_deposit_amount': 'nan', 'maximum_deposit_amount': 'nan', 'minimum_loan_amount': 100000000, 'guarantee': 0, 'receiving_channel': 'غیر حضوری', 'loan_amount': 100000000, 'deposit_amount': 100000, 'repayment_amount': 2819047}
{'id': 1, 'nickname': 'بهان(بدون ضامن)', 'package_name': 'فرابانک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 1000000000, 'deposit_duration': 3, 'interest_rate': 23, 'repayment_duration': 60, 'loan_coefficient': 1000, 'credit_score': 'A', 'minimum_deposit_amount': 'nan', 'maximum_deposit_amount': 'nan', 'minimum_loan_amount': 100000000, 'guarantee': 0, 'receiving_channel': 'غیر حضوری', 'loan_amount': 100000000, 'deposit_amou

In [86]:


def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                loan_amount=None, credit_score=None, interest_rate=None, 
                repayment_duration=None):

    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }


  
    updated_loans = fill_calculated_param (loans, loan_amount, deposit_amount)

   
    filtered_loans = []
    for loan in updated_loans:
        conditions = []

       
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                
                elif key == "deposit_amount":

                    # conditions.append(lower_bound <= loan.get(key) <= upper_bound)
                    calculated_deposit = loan.get("deposit_amount",0)
                    lower_bound = value * 0.1  
                    upper_bound = value * 20  
                    

            # calculated_deposit = loan.get("deposit_amount")
            # if calculated_deposit is not None:

                    if calculated_deposit is not None:
                        conditions.append(lower_bound <= calculated_deposit <= upper_bound)

                    else:
                        conditions.append(False)
          



                else:
                   
                    conditions.append(loan.get(key) == value)

      
        if conditions and all(conditions):
            filtered_loans.append(loan)

    return filtered_loans , loan_amount


In [ ]:
# filtered = filter_loans(loans, loan_amount= 100_000_000, deposit_amount= 120_000)
# filtered = filter_loans(loans, deposit_amount= 120000)
# filtered = filter_loans(loans, loan_amount= 2_100_000_000, interest_rate= 23)
filtered, loan_value = filter_loans(loans, loan_amount= 90_000_000 ,credit_score='A')
# filtered = filter_loans(loans, credit_score= "B")

print(len(filtered))
# print("Filtered loans:")
# # print(filtered[1])
# for loan in filtered:
#     print(loan)

5


In [94]:
loan_value

60000000

### Define sorting scores

In [95]:
filtered_df= pd.DataFrame(filtered)
df_sorting =filtered_df.copy()

IR_default   = {4:0.323, 23:0.6669, 18:0.005, 14:0.0049 }
RD_default = {60:0.0697, 48:0.0234, 36:0.2116, 24:0.3418, 12:0.3533}
DD_default  = {1:0.25, 2:0.2, 3:0.2, 4:0.15, 6:0.1, 12:0.1}
CS_default  = {'B':0.25,'A':0.20,'C':0.175,'N':0.175,'D':0.1,'E':0.1}

w = {
    "IR_score":0.3, "CS_score":0.30, "RD_score":0.20,
    "DD_score":0.20,
}
#  "DA_score":0.03, "LA_score":0.02,
# w_type = {"شایان": 1 , "نیک‌وام" : 2, "فرابانک" : 2, "بهان" : 1}
w_type_default = {"شایان": 0.14 , "نیک‌وام" : 0.78, "فرابانک" : 0.029, "بهان" : 0.047}

## consider loan amount values to separate weights

In [96]:



# for loan amount 1-50
IR_1_50   = {23:0.33, 4:0.313, 18:0.179, 14:0.178}
RD_1_50 = {12:0.22, 24:0.22, 36:0.19, 48:0.16, 60:0.21}
w_type_1_50 = {"شایان": 0.21 , "نیک‌وام" : 0.32, "فرابانک" : 0.2, "بهان" : 0.27}
##--------------------------------------------------------
# for loan amount 50-100
IR_50_100   = {23:0.593, 4:0.25, 18:0.081, 14:0.08}
RD_50_100 = {12:0.2, 24:0.22, 36:0.24, 48:0.11, 60:0.23}
w_type_50_100 = {"شایان": 0.18 , "نیک‌وام" : 0.34, "فرابانک" : 0.18, "بهان" : 0.3}

##--------------------------------------------------------
# for loan amount 100-150
IR_100_150   = {4:0.43, 23:0.36, 18:0.21, 14:0.03}
RD_100_150 = {12:0.24, 24:0.25, 36:0.21, 48:0.15, 60:0.15}
w_type_100_150 = {"شایان": 0.32 , "نیک‌وام" : 0.43, "فرابانک" : 0.24, "بهان" : 0.01}
##-------------------------------------------------------
# for loan amount 150-200
IR_150_200   = {23:0.33, 4:0.32, 18:0.175, 14:0.174}
RD_150_200 = {12:0.23, 24:0.24, 36:0.22, 48:0.15, 60:0.16}
w_type_150_200 = {"شایان": 0.35 , "نیک‌وام" : 0.39, "فرابانک" : 0.25, "بهان" : 0.01}
##-------------------------------------------------------
# for loan amount 200-250
IR_200_250   = {4:0.294, 23:0.277, 18:0.21, 14:0.21}
RD_200_250 = {12:0.26, 24:0.28, 36:0.24, 48:0.21, 60:0.01}
w_type_200_250 = {"شایان": 0.48 , "نیک‌وام" : 0.51, "فرابانک" : 0.005, "بهان" : 0.005}
##--------------------------------------------------------
# for loan amount 250-300
IR_250_300   = {4:0.355, 23:0.346, 18:0.15, 14:0.149}
RD_250_300 = {12:0.33, 24:0.26, 36:0.24, 48:0.16, 60:0.01}
w_type_250_300 = {"شایان": 0.49 , "نیک‌وام" : 0.5, "فرابانک" : 0.005, "بهان" : 0.005}

###########################################

def check_loan_amount(loan_amount, IR_default, RD_default,w_type_default):
    if loan_amount <= 50_000_000 :
        IR = IR_1_50
        RD = RD_1_50
        w_type = w_type_1_50
    elif 50_000_000< loan_amount <= 100_000_000:
        IR = IR_50_100
        RD = RD_50_100
        w_type = w_type_50_100
    elif 100_000_000< loan_amount <= 150_000_000:
        IR = IR_100_150
        RD = RD_100_150
        w_type = w_type_100_150
    elif 150_000_000< loan_amount <= 200_000_000:
        IR = IR_150_200
        RD = RD_150_200
        w_type = w_type_150_200
    elif 200_000_000< loan_amount <= 250_000_000:
        IR = IR_200_250
        RD = RD_200_250
        w_type = w_type_200_250
    elif 250_000_000< loan_amount :
        IR = IR_250_300
        RD = RD_250_300
        w_type = w_type_250_300
    else:
        IR = IR_default
        RD = RD_default
        w_type = w_type_default

    return IR, RD , w_type



### Calculate sorting scores

In [97]:
IR_MAP, RD_MAP , _ = check_loan_amount(loan_value, IR_default, RD_default, w_type_default)

In [98]:
IR_MAP, RD_MAP

({23: 0.593, 4: 0.25, 18: 0.081, 14: 0.08},
 {12: 0.2, 24: 0.22, 36: 0.24, 48: 0.11, 60: 0.23})

In [101]:
IR, RD, w_type = check_loan_amount(loan_value, IR_default, RD_default, w_type_default)
df_sorting["IR_score"]   = df_sorting["interest_rate"].map(IR)
df_sorting["RD_score"]   = df_sorting["repayment_duration"].map(RD)
df_sorting["DD_score"]   = df_sorting["deposit_duration"].map(DD_default)
df_sorting["CS_score"]   = df_sorting["credit_score"].map(CS_default).fillna(0)
df_sorting["type_score"] = df_sorting["nickname"].map(w_type)
# # optional continuous extras (scale 0‑1)
# df["DepAmt_score"] = df["deposit_amount"]/df["deposit_amount"].max()
# df["LoanAmt_score"] = df["loan_amount_limit"]/df["loan_amount_limit"].max()




df_sorting["Score"] = sum(w[c]*df_sorting[c] for c in w)
df_sorting["Score_new"] = df_sorting["type_score"]*df_sorting["Score"]


df_sorted = df_sorting.sort_values("Score_new", ascending=False)
# print(df_sorted[["id","Score"]].head())

In [102]:
df_sorted

,id,nickname,package_name,contract_type,granted_method,loan_amount_limit,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,minimum_deposit_amount,maximum_deposit_amount,minimum_loan_amount,guarantee,receiving_channel,loan_amount,deposit_amount,repayment_amount,IR_score,RD_score,DD_score,CS_score,type_score,Score,Score_new
2,92,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,3,23,36,30,D,nan,15000000000,100000000,1,حضوری,60000000,2000000,2322583,0.593,0.24,0.2,0.1,0.18,0.2959,0.053262
1,91,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,3,23,24,45,D,nan,15000000000,100000000,1,حضوری,60000000,1333333,3142398,0.593,0.22,0.2,0.1,0.18,0.2919,0.052542
0,90,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,3,23,12,110,D,nan,15000000000,100000000,1,حضوری,60000000,545455,5644579,0.593,0.20,0.2,0.1,0.18,0.2879,0.051822
4,94,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,6,23,36,70,D,nan,15000000000,100000000,1,حضوری,60000000,857143,2322583,0.593,0.24,0.1,0.1,0.18,0.2759,0.049662
3,93,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,6,23,24,100,D,nan,15000000000,100000000,1,حضوری,60000000,600000,3142398,0.593,0.22,0.1,0.1,0.18,0.2719,0.048942


In [68]:
len(df_sorted)

70